# Notebook 03 - Inflation analysis (no inflation in B&F)

Decomposes the oil-shock response into a price and a quantity part and computes the
Domar-weighted mean price (the only 'inflation' object in the static B&F model).

CAVEAT: the `eta_price_insensitivity` helper is intentionally a no-op for the fixed-L
B&F model (there is no eta parameter here); the real eta-variance result lives in the
main BeyondHulten package. It is kept only to document the negative finding.

In [2]:
# --- Project setup (robust path resolution) ---
# @__DIR__ resolves to this notebook's directory; we anchor on the package root.
using LinearAlgebra, Statistics, Printf, DelimitedFiles

const NOTEBOOK_DIR = @__DIR__
const REP_DIR = joinpath(NOTEBOOK_DIR, "..")          # bf_replication/
cd(REP_DIR)
include(joinpath(REP_DIR, "src", "BFReplication.jl"))
using .BFReplication
using .BFReplication.DataLoader
using .BFReplication.BFModel
using .BFReplication.InflationAnalysis

const DATA_DIR = joinpath(REP_DIR, "..", "Replication Files", "GDP Simulatin -- 88 Sector")
const RESULTS_DIR = joinpath(REP_DIR, "data", "results")
mkpath(RESULTS_DIR)

println("Project dir : ", REP_DIR)
println("Data dir    : ", DATA_DIR)
println("Results dir : ", RESULTS_DIR)

# --- Load data ---
data = load_bf_data(joinpath(DATA_DIR, "BFdata.csv"); year=1980)
describe_data(data)

Project dir : /Users/joko/Git/BFRep/(3)BeyondHulten/bf_replication/notebooks/..
Data dir    : /Users/joko/Git/BFRep/(3)BeyondHulten/bf_replication/notebooks/../../Replication Files/GDP Simulatin -- 88 Sector
Results dir : /Users/joko/Git/BFRep/(3)BeyondHulten/bf_replication/notebooks/../data/results
B&F Replication Data Summary
Sectors: 76
Base year: 1980

Ω matrix: 76x76 (cost shares)
α (factor shares): mean=0.4954, min=0.0989, max=0.8264
β (consumption shares): sum=1.0000, max sector=0.1416
λ (Domar weights): sum=2.0918, max=0.1551
L (labor allocation): sum=1.0000

Key checks:
  λ ⊙ α ≈ L? max diff = 0.00e+00
  (I - diag(1-α)·Ω)⁻¹' · β ≈ λ? max diff = 0.00e+00
  β sums to 1? sum = 1.0000000000


## Baseline vs oil shock: price vs quantity

In [3]:
# analyze_price_vs_quantity takes (BFParameters, BFParameters, sector)
A_base = ones(data.N)
A_shock = ones(data.N); A_shock[7] = 0.7
p_base = BFParameters(A_base, data.Ω, data.α, data.β, data.L, 0.5, 0.0001, 0.9)
p_shock = BFParameters(A_shock, data.Ω, data.α, data.β, data.L, 0.5, 0.0001, 0.9)

pq = analyze_price_vs_quantity(p_base, p_shock, 7)
println("oil-shock log price change    = $(pq.mean_price_change)")
println("oil-shock log quantity change = $(pq.real_gdp_change)")

oil-shock log price change    = 0.12920885010346675
oil-shock log quantity change = 0.0


## Inflation measures (Domar-weighted mean price)

In [4]:
# compute_inflation_measures takes (sol_baseline, sol_shocked, params)
sol_base = BFModel.compute_equilibrium(p_base)
sol_shock = BFModel.compute_equilibrium(p_shock)

measures = compute_inflation_measures(sol_base, sol_shock, p_shock)
println("mean log price change (Domar-weighted) = $(measures.mean_price_change)")

mean log price change (Domar-weighted) = 0.12920885010346675


## eta price insensitivity + network decomposition (see caveat above)

In [6]:
# eta_price_insensitivity takes (data, sector)
eta = eta_price_insensitivity(data, 7)
println("price CV to eta = $(eta.cpi_cv)   (no eta in fixed-L B&F -> no-op)")

# network_price_decomposition takes (params)
net = network_price_decomposition(p_shock, 7)
println("network decomposition: ", net)

price CV to eta = 0.0   (no eta in fixed-L B&F -> no-op)
network decomposition: (direct = 1.4972463027733642, upstream = 0.24448651921728912, downstream = 0.47482455869995527, general_equilibrium = -0.07470177647660217, total_cpi = 0.0, total_mean_price = 0.12920885010346675, all_price_changes = [-0.018356016824883692, -0.40936284941481865, -0.05770582308408504, -0.6189014475005367, 0.05230093811015934, -0.4626177297094234, 1.4972463027733642, -0.01977603318461408, -0.05558315757912279, -0.03265683190666044, 0.04496001414323659, -0.049460989418216944, -0.10766276553816849, -0.08175071109362354, -0.06795362373148083, -0.11443857079883653, -0.06920825707213799, -0.06661573022690132, -0.07046652650498722, -0.10807329156224962, -0.046782735594405024, -0.06861744364867683, -0.07947800711009993, -0.0948385001486283, -0.0707128761344112, -0.05027314249173804, -0.08205385193184363, -0.04488829314728675, -0.023405116242632782, -0.055388480547459436, 0.008993796938738044, -0.03499934621026377, -